# Retrieval from ChromaDB

Queries the ChromaDB collection built by `notebooks/ingestion_preprocessing_embedding.ipynb`.

**Run the ingestion notebook first** so `notebooks/chroma_db/` exists on disk.

In [1]:
!pip install chromadb transformers torch


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import chromadb
from transformers import AutoTokenizer, AutoModel
import torch

client = chromadb.PersistentClient(path="chroma_db")
collection = client.get_collection("actas_obra")
print(f"Collection loaded with {collection.count()} chunks.")

c:\REPOS\ML problems\POSTGRAU\RAG-UPCSchool-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Collection loaded with 662 chunks.


In [3]:
model_name = "BSC-LT/MrBERT-es"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

Loading weights: 100%|██████████| 134/134 [00:00<00:00, 16089.45it/s]
[transformers] ModernBertModel LOAD REPORT from: BSC-LT/MrBERT-es
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
def embed_query(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze().tolist()

In [5]:
search_query = "¿Cuál es el estado actual de la gestión del informe de conexión de acometida pluvial con Drenajes Besós?"

query_embedding = embed_query(search_query)

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
)):
    print(f"--- Rank {i+1} | Chunk ID: {meta['chunk_id']} | Distance: {dist:.4f} ---")
    print(doc)
    print()

--- Rank 1 | Chunk ID: 404 | Distance: 590.4879 ---
Por lo tanto, se descartaba el material de PE que proponía la constructora. Renfe comunica tras consultar con el dpto. de Autoprotección que el material de la instalación de la columna seca debe ser acero. Respecto a si debe ir visto o enterrado, es mejor que vaya visto por un tema de mantenimiento, solamente se colocará enterrado en zonas puntuales si es necesario. Por lo tanto, el trazado por el andén será preferiblemente visto.

--- Rank 2 | Chunk ID: 200 | Distance: 608.9624 ---
La máquina está colgada mediante varillas roscadas, número de unidades:4. La máquina descansa parcialmente sobre perfiles de falso techo, que NO están diseñados para soportar peso (solo sujetan placas). Se ha de definir: 4 varillas M8/M10 ancladas al forjado. Definir diámetro de varillas instaladas. 2. Colocar una bancada metálica o marco perimetral. Estudiar esta opción. Puede ser: Omegas reforzadas Perfil de acero galvanizado 40×40 Cuadro de pletinas sol